# W05｜Transaction / Isolation / Lock / Pool｜教学版

**建议学习时间：6h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 Isolation/Lock/Pool/Persistence | DB Engineering + LangGraph Persistence | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Checkpoint / Resume | 知道 checkpoint 不等于 DB transaction | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 模拟 2 个并发请求更新同一 Job | 比较无保护 vs optimistic lock | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 给 query_job 加 status + version | 并发更新不会静默覆盖 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 回答 2 个生产问题 | 500 请求≠500 DB连接；worker并发更新怎么办 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜并发更新验收
无保护：后写覆盖先写；Optimistic Lock：旧 version 的第二次更新返回 affected rows = 0。

### Task 4｜参考 SQL
```sql
UPDATE query_job
SET status=:new_status, version=version+1
WHERE id=:id AND version=:expected_version;
```

### Task 5｜生产问题参考答案
- 500 HTTP requests ≠ 500 DB connections：应用通过有限 pool 复用连接；
- 多 worker 更新同一 job：用 version/lock + 明确状态转移，冲突必须检测，不能静默覆盖。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解典型并发异常
- 会解释并实现 optimistic locking
- 会看基础 EXPLAIN
- 理解 connection pool

        ## 本周前置检查

        - 完成 W04；理解 PK/FK/Session

        ## 本周必须记住的英文术语

        - `transaction isolation`：事务隔离
- `lost update`：丢失更新
- `optimistic lock`：乐观锁
- `deadlock`：死锁
- `connection pool`：连接池
- `EXPLAIN`：执行计划

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. Lost Update 与 Isolation

    **是什么（What）**  
    两个并发事务都读到旧值，然后后提交的更新覆盖先提交的结果，就可能发生 Lost Update。

    **为什么需要（Why）**  
    Production 中 job/status、余额、库存、计数器都可能被并发修改。

    **什么时候用（When）**  
    并发写业务状态时必须考虑 transaction isolation / locking / version。

    **常见误区 / 容易踩坑**  
    有 transaction 不代表自动没有并发问题；Isolation Level 和更新方式非常关键。


**中英术语**：`transaction` = 事务；`isolation level` = 隔离级别；`lost update` = 丢失更新；`dirty read` = 脏读；`phantom read` = 幻读

### 代码实验：先模拟 Lost Update

**这段代码想让你看什么？**

两个请求都基于 version=1；B 不知道 A 已经更新。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
record = {"status": "RUNNING", "version": 1}

a_snapshot = record.copy()
b_snapshot = record.copy()

record["status"] = "DONE"    # A 更新
record["version"] += 1

record["status"] = "FAILED"  # B 仍基于旧认知覆盖
record["version"] += 1

print(record)

## 2. Optimistic Lock

    **是什么（What）**  
    每条记录带 version；更新时要求 `WHERE version = expected_version`，成功后 version+1。

    **为什么需要（Why）**  
    适合冲突不是特别频繁、又不想长期持有数据库锁的场景。

    **什么时候用（When）**  
    Job 状态、配置更新、表单版本。

    **常见误区 / 容易踩坑**  
    Optimistic lock 不是“没有锁”，而是冲突检测策略；更新失败后要决定 retry、重新读取或提示冲突。


**中英术语**：`optimistic lock` = 乐观锁；`version column` = 版本列；`affected rows` = 受影响行数

### 代码实验：为什么第二个更新会失败

**这段代码想让你看什么？**

A 成功后 version 变 2；B 仍期待 version=1，因此应被拒绝。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
record = {"status": "RUNNING", "version": 1}

def optimistic_update(expected_version, new_status):
    if record["version"] != expected_version:
        return False
    record["status"] = new_status
    record["version"] += 1
    return True

expected = record["version"]
print("A:", optimistic_update(expected, "DONE"))
print("B:", optimistic_update(expected, "FAILED"))
print(record)

## 3. Index / Query Planner

    **是什么（What）**  
    Index 是辅助数据结构；Query Planner 决定具体执行计划。

    **为什么需要（Why）**  
    加了 index 不代表一定会用；低选择性、小表、统计信息等都会影响 planner。

    **什么时候用（When）**  
    分析慢 SQL、过滤/Join/排序热点。

    **常见误区 / 容易踩坑**  
    不要看到慢查询就无脑加 index；index 也有写入和存储成本。


**中英术语**：`index` = 索引；`query planner` = 查询规划器；`EXPLAIN` = 执行计划解释；`selectivity` = 选择性

In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE event(id INTEGER PRIMARY KEY, channel TEXT, month TEXT, volume INTEGER)")
rows = [(i, ["SMS","EMAIL","PUSH"][i%3], f"2026-{(i%12)+1:02d}", i%100)
        for i in range(1, 20001)]
cur.executemany("INSERT INTO event VALUES (?, ?, ?, ?)", rows)

print("Before index:")
print(cur.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM event WHERE channel=? AND month=?",
    ("SMS","2026-06")
).fetchall())

cur.execute("CREATE INDEX idx_event_channel_month ON event(channel, month)")

print("After index:")
print(cur.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM event WHERE channel=? AND month=?",
    ("SMS","2026-06")
).fetchall())

## 4. Connection Pool

    **是什么（What）**  
    应用通常不会为每个请求永久建立一个新数据库连接，而是从有限连接池借用连接。

    **为什么需要（Why）**  
    数据库连接成本高，数据库服务也有连接上限。

    **什么时候用（When）**  
    FastAPI/Worker 的数据库访问。

    **常见误区 / 容易踩坑**  
    Pool 太小会排队；太大可能把数据库压垮。HTTP 并发数 ≠ DB pool size。


**中英术语**：`connection pool` = 连接池；`pool exhaustion` = 连接池耗尽；`queue wait` = 排队等待

### 练习：写出真正的 Optimistic Lock SQL

把上面的逻辑写成 SQL 思路：根据 `id + expected_version` 更新 status 和 version。

<details>
    <summary><b>参考答案</b></summary>

    ```python
UPDATE query_job
SET status = :new_status,
    version = version + 1
WHERE id = :id
  AND version = :expected_version;

-- affected rows = 0 时表示版本冲突或记录不存在
```

    </details>

### Checkpoint

1. 为什么慢 LLM 调用不应该包在一个长数据库 transaction 里？
2. Pool 从 20 调到 200 为什么不一定更快？
3. Checkpoint 和 Transaction 是同一件事吗？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 会长期占用连接和锁，扩大冲突/超时范围；通常应缩短 transaction，把远程 IO 放在 transaction 之外。

**2.** 数据库 CPU/IO/锁/最大连接仍有限，更多连接可能增加竞争和上下文开销。

**3.** 不是。Checkpoint 记录 workflow 执行状态；Transaction 保护数据库操作的一致性/原子性。

    </details>

## ChatBI 本周作业

        给 `query_job` 加 `version`，实现状态更新冲突检测。

在 README 回答：
1. 为什么 LLM 调用不放在长 transaction 内；
2. 为什么 API 500 并发不等于 DB pool=500。

### 本周面试 / 理解检查

1. 什么是 Lost Update？
2. 乐观锁适合什么场景？
3. 为什么 connection pool 不是越大越好？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 并发请求基于同一个旧版本修改，后写结果覆盖先写结果。

**2.** 冲突不频繁、希望减少长期锁持有的场景；冲突时通过 version 检测。

**3.** 数据库资源有限；过多连接会增加资源占用、锁竞争和调度开销。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。